# Introduction to Jupyter, Python, Pandas, and Polars for SQL Students
##### Professor: Joanna
##### Developed by: Ariana Ghimire

<hr style="border: 2px solid #003262">

<hr style="border: 2px solid #C9B676">

## Welcome

You've spent about eight weeks learning **SQL** — how to retrieve data with `SELECT`, filter with `WHERE`, summarize with `GROUP BY`, and combine tables with `JOIN`.

This notebook will bridge your SQL knowledge into Python data analysis.

**Pandas is the primary library for this module.** Polars examples are included as a **modern comparison**. You are **not** expected to memorize both syntaxes.

We use the **AP (Accounts Payable)** database from your SQL course — vendors, invoices, payment terms, and general-ledger accounts — stored as CSV files in this project folder.

## Learning Objectives

By the end of this notebook, you should be able to:

- Navigate and run cells in a Jupyter notebook
- Load tabular data into **DataFrames** with **Pandas** (primary)
- Translate common SQL operations into **Pandas** syntax
- Recognize **Polars** as a modern alternative (optional comparison — you do not need to memorize it)
- Use advanced SQL ideas in **Pandas**: **outer joins**, **set operations**, **subqueries**, and **hierarchical (recursive) data**
- Explain why Python is commonly used **after** SQL in data workflows

> **How to use this notebook:** Read each short section, run the code cell immediately below it, and try the practice question before expanding the **Solution** section to check your work.

> **Pandas vs Polars:** Complete all **Practice** exercises in **Pandas**. Polars cells and any prompt marked **Try this in Polars (optional)** are for curiosity and comparison only — skip them if you want to focus on one Python library.

<hr style="border: 2px solid #003262">

## Table of Contents

1. [Jupyter Basics](#1-jupyter-basics)
2. [Python Basics](#2-python-basics)
3. [Loading Data](#3-loading-data)
4. [SQL to Pandas](#4-sql-to-pandas)
5. [SQL to Polars](#5-sql-to-polars) *(optional comparison)*
6. [Advanced SQL Translations](#6-advanced-sql-translations)
   - [6.1 Outer Joins](#61-outer-joins)
   - [6.2 Set Operations](#62-set-operations)
   - [6.3 Subqueries](#63-subqueries)
   - [6.4 Recursive CTEs](#64-recursive-ctes)
7. [Practice](#7-practice)
8. [Reference Guide](#8-reference-guide)

> Tip: In Jupyter, you can also use the **Table of Contents** sidebar / outline if your interface provides one.

<hr style="border: 2px solid #003262">

## 1. Jupyter Basics

A **Jupyter notebook** mixes explanatory text (Markdown) with runnable **code cells**. Each cell runs independently, which makes exploration easy.

| Cell type | Purpose |
|-----------|----------|
| **Markdown** | Headings, explanations, tables (like this cell) |
| **Code** | Python you can run and re-run |

**Running a cell:** Click a cell and press `Shift + Enter` (runs and moves down) or `Ctrl/Cmd + Enter` (runs in place).

**Restart & Run All:** `Kernel → Restart & Run All` re-runs every cell from the top — useful after editing earlier cells.

**Useful shortcuts:**
- `A` — insert cell above (command mode)
- `B` — insert cell below
- `D, D` — delete cell
- `M` — change cell to Markdown
- `Y` — change cell to Code

**Exercise:** Run the cell below. You should see a greeting printed.

In [ ]:
print("Hello from Jupyter! You ran your first cell.")

<hr style="border: 2px solid #003262">

## 2. Python Basics

We only cover Python concepts you'll use in this notebook. No loops, classes, or file writing.

## Variables

A **variable** stores a value under a name.

In [ ]:
x = 10
y = 3.5
name = "AP Database"
print(x, y, name)

## Strings and Lists

**Strings** hold text. **Lists** hold ordered collections.

In [ ]:
vendor = "Federal Express Corporation"
states = ["CA", "WI", "TN"]
print(vendor.upper())
print(len(states))
print(states[0])

## Basic Math

In [ ]:
invoice_total = 3813.33
tax = invoice_total * 0.08
print(invoice_total + tax)
print(invoice_total / 2)

## Built-in Functions

`print()`, `len()`, and `type()` appear often:

In [ ]:
print(type(invoice_total))
print(len("Invoices"))

## Importing Libraries

> **Kernel check:** If imports fail, use **Kernel → Change Kernel** and select **Python (ecc-csci)** or your project's `.venv` interpreter — not the system Python.

**Pandas** and **Polars** are imported with standard aliases:

In [ ]:
import pandas as pd
import polars as pl
import numpy as np

print(pd.__version__)

**Practice:** Create a variable `terms_days = 30` and print its type.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
terms_days = 30
print(type(terms_days))
```

</details>

In [ ]:
# Your code here


<hr style="border: 2px solid #003262">

## 3. Loading Data

In SQL, data lives in **tables**. In Python, the equivalent is a **DataFrame** — a table with **rows**, **columns**, **column names**, and **data types**.

```
Database Table  →  DataFrame
     row          →  row (observation)
   column         →  column (variable)
```

Our dataset comes from the AP database (`Vendors`, `Invoices`, `Terms`, `GLAccounts`, etc.), exported to CSV files in this folder.

## Load with Pandas

In [ ]:
vendors_pd = pd.read_csv("Vendors.csv")
invoices_pd = pd.read_csv("Invoices.csv")
terms_pd = pd.read_csv("Terms.csv")
vendors_pd.head()

## Load with Polars

In [ ]:
vendors_pl = pl.read_csv("Vendors.csv")
invoices_pl = pl.read_csv("Invoices.csv")
terms_pl = pl.read_csv("Terms.csv")
vendors_pl.head()

## Inspecting a DataFrame

In [ ]:
print("Shape (rows, columns):", vendors_pd.shape)
print("Columns:", list(vendors_pd.columns))
vendors_pd.dtypes

**Practice:** Load `GLAccounts.csv` into a Pandas DataFrame called `gl_pd` and display its first 5 rows.

<details>
<summary><strong>Solution:</strong></summary>

```python
gl_pd = pd.read_csv("GLAccounts.csv")
gl_pd.head()
```

</details>

In [ ]:
# Your code here


<hr style="border: 2px solid #003262">

## SQL Review

This is a **refresher**, not a re-teach. We'll use the AP schema throughout.

**Key tables:**
- `Vendors` — companies we pay (`VendorID`, `VendorName`, `VendorState`, …)
- `Invoices` — bills received (`InvoiceID`, `VendorID`, `InvoiceTotal`, `TermsID`, …)
- `Terms` — payment terms (`TermsID`, `TermsDescription`, `TermsDueDays`)
- `GLAccounts` — chart of accounts (`AccountNo`, `AccountDescription`)

| SQL clause | What it does | AP example |
|------------|--------------|------------|
| `SELECT` | Choose columns | vendor names |
| `FROM` | Choose table | `Vendors` |
| `WHERE` | Filter rows | invoices over 10,000 |
| `ORDER BY` | Sort results | largest invoices first |
| `GROUP BY` | Group rows | count vendors per state |
| `COUNT`, `AVG`, `MIN`, `MAX` | Summarize | average invoice total |
| `JOIN` | Combine tables | invoices + vendor names |

**Example SQL** (from the AP database):

```sql
SELECT VendorName, VendorState
FROM Vendors
WHERE VendorState = 'CA'
ORDER BY VendorName;
```

```sql
SELECT TermsID, AVG(InvoiceTotal) AS AvgTotal
FROM Invoices
GROUP BY TermsID;
```

```sql
SELECT v.VendorName, i.InvoiceTotal
FROM Invoices i
JOIN Vendors v ON i.VendorID = v.VendorID
WHERE i.InvoiceTotal > 20000;
```

<hr style="border: 2px solid #003262">

## 4. SQL to Pandas

For each SQL concept: **SQL → Pandas → explanation → practice**.

We'll use `vendors_pd` and `invoices_pd` loaded earlier.

## Selecting Columns (`SELECT`)

**SQL:**
```sql
SELECT VendorName, VendorCity FROM Vendors;
```

**Pandas:** Use double brackets `[[]]` or `.loc` to select columns.

In [ ]:
vendors_pd[["VendorName", "VendorCity"]].head()

**Practice:** Select `InvoiceID`, `InvoiceTotal`, and `VendorID` from `invoices_pd` (first 5 rows).

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pd[["InvoiceID", "InvoiceTotal", "VendorID"]].head()
```

</details>

In [ ]:
# Your code here


## Filtering Rows (`WHERE`)

**SQL:**
```sql
SELECT * FROM Invoices WHERE InvoiceTotal > 10000;
```

**Pandas:** Boolean indexing — pass a condition inside `[]`.

In [ ]:
invoices_pd[invoices_pd["InvoiceTotal"] > 10000].head()

**Practice:** Show vendors in California (`VendorState == 'CA'`). Display `VendorName` and `VendorCity`.

<details>
<summary><strong>Solution:</strong></summary>

```python
vendors_pd[vendors_pd["VendorState"] == "CA"][["VendorName", "VendorCity"]].head()
```

</details>

In [ ]:
# Your code here


## Sorting (`ORDER BY`)

**SQL:**
```sql
SELECT VendorName, InvoiceTotal
FROM Invoices i JOIN Vendors v ...
ORDER BY InvoiceTotal DESC;
```

**Pandas:** `.sort_values()` with `ascending=False` for descending.

In [ ]:
invoices_pd.sort_values("InvoiceTotal", ascending=False).head()

**Practice:** Sort `vendors_pd` by `VendorName` alphabetically. Show the first 5 rows.

<details>
<summary><strong>Solution:</strong></summary>

```python
vendors_pd.sort_values("VendorName").head()
```

</details>

In [ ]:
# Your code here


## Creating New Columns

**SQL:**
```sql
SELECT InvoiceTotal, PaymentTotal,
       InvoiceTotal - PaymentTotal AS BalanceDue
FROM Invoices;
```

**Pandas:** Assign to a new column name with `df["new_col"] = ...`

In [ ]:
sample = invoices_pd[["InvoiceTotal", "PaymentTotal"]].copy()
sample["BalanceDue"] = sample["InvoiceTotal"] - sample["PaymentTotal"]
sample.head()

**Practice:** Add a column `IsLarge` to a copy of `invoices_pd` that is `True` when `InvoiceTotal > 5000`.

<details>
<summary><strong>Solution:</strong></summary>

```python
inv_copy = invoices_pd.copy()
inv_copy["IsLarge"] = inv_copy["InvoiceTotal"] > 5000
inv_copy[["InvoiceID", "InvoiceTotal", "IsLarge"]].head()
```

</details>

In [ ]:
# Your code here


## Aggregate Functions (`COUNT`, `AVG`, `MIN`, `MAX`)

**SQL:**
```sql
SELECT AVG(InvoiceTotal), MAX(InvoiceTotal), MIN(InvoiceTotal)
FROM Invoices;
```

**Pandas:** Call `.mean()`, `.max()`, `.min()`, `.count()` on a column.

In [ ]:
print("Average:", invoices_pd["InvoiceTotal"].mean())
print("Max:", invoices_pd["InvoiceTotal"].max())
print("Min:", invoices_pd["InvoiceTotal"].min())
print("Count:", invoices_pd["InvoiceTotal"].count())

**Practice:** What is the average `TermsDueDays` in `terms_pd`?

<details>
<summary><strong>Solution:</strong></summary>

```python
terms_pd["TermsDueDays"].mean()
```

</details>

In [ ]:
# Your code here


## Group By (`GROUP BY`)

**SQL:**
```sql
SELECT VendorState, COUNT(*) AS NumVendors
FROM Vendors
GROUP BY VendorState
ORDER BY NumVendors DESC;
```

**Pandas:** `.groupby("col").agg(...)` or `.groupby("col")["col2"].mean()`

In [ ]:
vendors_pd.groupby("VendorState").size().sort_values(ascending=False).head(10)

In [ ]:
invoices_pd.groupby("TermsID")["InvoiceTotal"].agg(["count", "mean", "max"]).round(2)

**Practice:** Group `invoices_pd` by `VendorID` and compute the sum of `InvoiceTotal`. Sort descending, show top 5.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pd.groupby("VendorID")["InvoiceTotal"].sum().sort_values(ascending=False).head()
```

</details>

In [ ]:
# Your code here


## Distinct (`DISTINCT` / `SELECT DISTINCT`)

**SQL:**
```sql
SELECT DISTINCT VendorState FROM Vendors;
```

**Pandas:** `.drop_duplicates()` or `.unique()` on a column.

In [ ]:
sorted(vendors_pd["VendorState"].unique())

In [ ]:
vendors_pd[["VendorState"]].drop_duplicates().sort_values("VendorState").head(10)

**Practice:** How many distinct `TermsID` values appear in `invoices_pd`?

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pd["TermsID"].nunique()
```

</details>

In [ ]:
# Your code here


## Head / Limit (`TOP` / `LIMIT`)

**SQL:**
```sql
SELECT TOP 5 VendorName FROM Vendors ORDER BY VendorName;
```

**Pandas:** `.head(n)` returns the first *n* rows (after sorting if needed).

In [ ]:
vendors_pd.sort_values("VendorName").head(5)[["VendorName"]]

**Practice:** Show the 3 invoices with the highest `InvoiceTotal`.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pd.nlargest(3, "InvoiceTotal")[["InvoiceID", "InvoiceTotal"]]
```

</details>

In [ ]:
# Your code here


## Joins (`JOIN`)

**SQL:**
```sql
SELECT v.VendorName, i.InvoiceNumber, i.InvoiceTotal
FROM Invoices i
INNER JOIN Vendors v ON i.VendorID = v.VendorID;
```

**Pandas:** `pd.merge(left, right, on="key")` or `how="inner"`.

In [ ]:
inv_vendors = pd.merge(
    invoices_pd,
    vendors_pd[["VendorID", "VendorName", "VendorState"]],
    on="VendorID",
    how="inner"
)
inv_vendors[["VendorName", "InvoiceNumber", "InvoiceTotal"]].head()

**Practice:** Join `invoices_pd` with `terms_pd` on `TermsID`. Show `InvoiceID`, `InvoiceTotal`, and `TermsDescription` for the first 5 rows.

<details>
<summary><strong>Solution:</strong></summary>

```python
pd.merge(invoices_pd, terms_pd, on="TermsID")[["InvoiceID", "InvoiceTotal", "TermsDescription"]].head()
```

</details>

In [ ]:
# Your code here


<hr style="border: 2px solid #003262">

## 5. SQL to Polars

**Optional comparison section.** Pandas remains the required library for this module.

Polars uses a **method-chaining** style that reads similarly to SQL. It is newer, often **faster on large datasets**, and increasingly used in modern data science (including newer Berkeley Data Science curriculum).

You do **not** need to memorize Polars syntax. Skim the examples, then attempt the **Try this in Polars (optional)** prompts only if you want extra practice.

## Selecting Columns — `select()`

```sql
SELECT VendorName, VendorState FROM Vendors;
```

In [ ]:
vendors_pl.select("VendorName", "VendorState").head()

**Try this in Polars (optional):** Select `InvoiceID` and `InvoiceTotal` from `invoices_pl`.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pl.select("InvoiceID", "InvoiceTotal").head()
```

</details>

In [ ]:
# Your code here


## Filtering — `filter()`

```sql
SELECT * FROM Invoices WHERE InvoiceTotal > 10000;
```

In [ ]:
invoices_pl.filter(pl.col("InvoiceTotal") > 10000).head()

**Try this in Polars (optional):** Filter vendors where `VendorState` is `'TN'`.

<details>
<summary><strong>Solution:</strong></summary>

```python
vendors_pl.filter(pl.col("VendorState") == "TN")
```

</details>

In [ ]:
# Your code here


## New Columns — `with_columns()`

```sql
SELECT InvoiceTotal - PaymentTotal AS BalanceDue FROM Invoices;
```

In [ ]:
invoices_pl.with_columns(
    (pl.col("InvoiceTotal") - pl.col("PaymentTotal")).alias("BalanceDue")
).select("InvoiceID", "InvoiceTotal", "PaymentTotal", "BalanceDue").head()

**Try this in Polars (optional):** Add a column `DoubleTotal` = `InvoiceTotal * 2`.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pl.with_columns(
    (pl.col("InvoiceTotal") * 2).alias("DoubleTotal")
).select("InvoiceID", "InvoiceTotal", "DoubleTotal").head()
```

</details>

In [ ]:
# Your code here


## Group By — `group_by()` + `agg()`

```sql
SELECT TermsID, AVG(InvoiceTotal) FROM Invoices GROUP BY TermsID;
```

In [ ]:
invoices_pl.group_by("TermsID").agg(
    pl.col("InvoiceTotal").mean().alias("AvgTotal"),
    pl.col("InvoiceID").count().alias("InvoiceCount")
)

**Try this in Polars (optional):** Count vendors per `VendorState`, sorted by count descending.

<details>
<summary><strong>Solution:</strong></summary>

```python
vendors_pl.group_by("VendorState").agg(
    pl.len().alias("NumVendors")
).sort("NumVendors", descending=True)
```

</details>

In [ ]:
# Your code here


## Sort — `sort()`

```sql
SELECT * FROM Invoices ORDER BY InvoiceTotal DESC;
```

In [ ]:
invoices_pl.sort("InvoiceTotal", descending=True).head(5)

**Try this in Polars (optional):** Sort `terms_pl` by `TermsDueDays` ascending.

<details>
<summary><strong>Solution:</strong></summary>

```python
terms_pl.sort("TermsDueDays")
```

</details>

In [ ]:
# Your code here


## Distinct — `unique()`

```sql
SELECT DISTINCT VendorState FROM Vendors;
```

In [ ]:
vendors_pl.select("VendorState").unique().sort("VendorState")

**Try this in Polars (optional):** Get unique `TermsID` values from `invoices_pl`.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pl.select("TermsID").unique()
```

</details>

In [ ]:
# Your code here


## Head — `head()`

```sql
SELECT TOP 5 * FROM Vendors;
```

In [ ]:
vendors_pl.head(5)

**Try this in Polars (optional):** Show the top 3 rows of `terms_pl`.

<details>
<summary><strong>Solution:</strong></summary>

```python
terms_pl.head(3)
```

</details>

In [ ]:
# Your code here


## Joins — `join()`

```sql
SELECT v.VendorName, i.InvoiceTotal
FROM Invoices i JOIN Vendors v ON i.VendorID = v.VendorID;
```

In [ ]:
invoices_pl.join(
    vendors_pl.select("VendorID", "VendorName"),
    on="VendorID",
    how="inner"
).select("VendorName", "InvoiceNumber", "InvoiceTotal").head()

**Try this in Polars (optional):** Join invoices with terms; show `InvoiceID`, `InvoiceTotal`, `TermsDescription`.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pl.join(terms_pl, on="TermsID").select(
    "InvoiceID", "InvoiceTotal", "TermsDescription"
).head()
```

</details>

In [ ]:
# Your code here


<hr style="border: 2px solid #003262">

## 6. Advanced SQL Translations

You already mapped basic SQL (`SELECT`, `WHERE`, `GROUP BY`, inner `JOIN`) to DataFrames.

This section covers **advanced SQL ideas** you likely saw near the end of your SQL course:

- Outer joins (`LEFT JOIN`, `RIGHT JOIN`, `FULL OUTER JOIN`)
- Set operations (`UNION`, `UNION ALL`, `INTERSECT`, `EXCEPT`)
- Subqueries
- Recursive CTEs for hierarchical data

> **Goal:** See each idea first as **SQL**, then as **pandas** (required). **Polars** side-by-side examples are optional comparison only. Use small example tables you can inspect by eye.

### 6.1 Outer Joins

Inner joins keep only matching keys. Outer joins also keep **unmatched** rows and fill missing side with `NULL`.

We will use tiny `employees` and `departments` tables — some IDs match, some do not.

In [ ]:
employees = pd.DataFrame(
    {
        "emp_id": [1, 2, 3, 4],
        "emp_name": ["Ada", "Lin", "Sam", "Kai"],
        "dept_id": [10, 20, 30, 40],  # 40 has no matching department
    }
)

departments = pd.DataFrame(
    {
        "dept_id": [10, 20, 50],  # 50 has no matching employee
        "dept_name": ["Sales", "Engineering", "Legal"],
    }
)

employees_pl = pl.DataFrame(employees.to_dict(orient="list"))
departments_pl = pl.DataFrame(departments.to_dict(orient="list"))

display(employees)
display(departments)

**SQL (conceptual)**

```sql
-- LEFT JOIN: keep all employees; unmatched dept columns are NULL
SELECT e.emp_name, d.dept_name
FROM employees e
LEFT JOIN departments d ON e.dept_id = d.dept_id;

-- RIGHT JOIN: keep all departments; unmatched emp columns are NULL
SELECT e.emp_name, d.dept_name
FROM employees e
RIGHT JOIN departments d ON e.dept_id = d.dept_id;

-- FULL OUTER JOIN: keep unmatched rows from both sides
SELECT e.emp_name, d.dept_name
FROM employees e
FULL OUTER JOIN departments d ON e.dept_id = d.dept_id;
```

| Join | Keeps unmatched rows from | Why nulls appear |
|------|---------------------------|------------------|
| `LEFT` | Left table (`employees`) | No matching `departments` row |
| `RIGHT` | Right table (`departments`) | No matching `employees` row |
| `FULL OUTER JOIN` | Both tables | Either side has no match |

**pandas** — `how="left"`, `how="right"`, and `how="outer"` map to SQL Server `LEFT JOIN`, `RIGHT JOIN`, and `FULL OUTER JOIN`.

In [ ]:
print("LEFT JOIN")
display(pd.merge(employees, departments, on="dept_id", how="left"))

print("RIGHT JOIN")
display(pd.merge(employees, departments, on="dept_id", how="right"))

print("FULL OUTER JOIN")
display(pd.merge(employees, departments, on="dept_id", how="outer"))

**Polars** — `how="left"` for `LEFT JOIN`; `how="full"` for `FULL OUTER JOIN`.

For teaching `RIGHT JOIN`, Polars does not need a special right join: reverse the tables and use a **left join**. Depending on your installed Polars version, `how="right"` may not be supported consistently — the reversed left join is safer and less version-dependent.

> A right join is equivalent to switching the table order and performing a left join.

In [ ]:
print("LEFT JOIN")
display(employees_pl.join(departments_pl, on="dept_id", how="left"))

print("RIGHT JOIN")
display(
    departments_pl.join(
        employees_pl,
        on="dept_id",
        how="left",
    )
)

print("FULL OUTER JOIN")
display(employees_pl.join(departments_pl, on="dept_id", how="full", coalesce=True))

**Check Your Understanding (prediction):** Before running code, predict which employee names appear with a missing `dept_name` in a **LEFT** join of `employees` to `departments`. Then verify with pandas.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
left_join = pd.merge(employees, departments, on="dept_id", how="left")
left_join[left_join["dept_name"].isna()][["emp_name", "dept_id", "dept_name"]]
# Sam (dept_id 30) and Kai (dept_id 40) have no matching departments, so dept_name is null.
```

</details>

In [ ]:
# Your code here


### 6.2 Set Operations

Set operations combine rows from two result sets with the **same columns**.

**Important:** `UNION`, `INTERSECT`, and `EXCEPT` compare the **complete selected row**, not only one column. For example, in our sample data `(4, "HIST")` is the duplicate row in `spring` — not simply student `4`.

For SQL set operations, both queries must return the **same number of columns in the same order**, with **compatible data types**.

- **`UNION ALL`** — stack all rows; keep duplicates
- **`UNION`** — stack rows; **remove duplicates**
- **`INTERSECT`** — rows that appear in **both**
- **`EXCEPT`** (SQL Server: `EXCEPT`) — rows in the first set **but not** the second

> **Key difference:** `UNION` drops duplicate rows; `UNION ALL` keeps them. `UNION ALL` is usually faster when you know duplicates are fine.

In [ ]:
fall = pd.DataFrame(
    {
        "student_id": [1, 2, 3, 4],
        "course": ["CSCI", "MATH", "CSCI", "HIST"],
    }
)

spring = pd.DataFrame(
    {
        "student_id": [3, 4, 4, 5],
        "course": ["CSCI", "HIST", "HIST", "CSCI"],  # (4, HIST) repeats; overlaps with fall
    }
)

fall_pl = pl.DataFrame(fall.to_dict(orient="list"))
spring_pl = pl.DataFrame(spring.to_dict(orient="list"))

print("Fall enrollments")
display(fall)
print("Spring enrollments")
display(spring)

**SQL (conceptual)**

```sql
SELECT student_id, course FROM fall
UNION ALL
SELECT student_id, course FROM spring;

SELECT student_id, course FROM fall
UNION
SELECT student_id, course FROM spring;

SELECT student_id, course FROM fall
INTERSECT
SELECT student_id, course FROM spring;

SELECT student_id, course FROM fall
EXCEPT
SELECT student_id, course FROM spring;
```

**pandas**

In [ ]:
union_all = pd.concat([fall, spring], ignore_index=True)
union_distinct = pd.concat([fall, spring], ignore_index=True).drop_duplicates()
intersect = pd.merge(
    fall.drop_duplicates(),
    spring.drop_duplicates(),
    on=["student_id", "course"],
    how="inner",
)

except_rows = (
    fall.drop_duplicates()
    .merge(
        spring.drop_duplicates(),
        on=["student_id", "course"],
        how="left",
        indicator=True,
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
)

print("UNION ALL"); display(union_all)
print("UNION"); display(union_distinct)
print("INTERSECT"); display(intersect)
print("EXCEPT (fall minus spring)"); display(except_rows)

**Polars**

In [ ]:
union_all_pl = pl.concat([fall_pl, spring_pl])
union_pl = pl.concat([fall_pl, spring_pl]).unique()
intersect_pl = (
    fall_pl.unique()
    .join(spring_pl.unique(), on=["student_id", "course"], how="inner")
)
except_pl = (
    fall_pl.unique()
    .join(spring_pl.unique(), on=["student_id", "course"], how="anti")
)

print("UNION ALL"); display(union_all_pl)
print("UNION"); display(union_pl)
print("INTERSECT"); display(intersect_pl)
print("EXCEPT"); display(except_pl)

**Set operations recap:** `UNION ALL` stacks rows; `UNION` drops duplicates; `INTERSECT` keeps rows in both sets; `EXCEPT` keeps rows only in the first set. Each operation compares the **complete selected row** — see **Section 8 — Reference Guide** for the pandas/Polars mapping.

**Practice:** Using `fall` and `spring`, create a **UNION ALL** result and count how many rows it has. Then create a **UNION** result and count those rows. How many duplicates were removed?

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
all_rows = pd.concat([fall, spring], ignore_index=True)
distinct_rows = all_rows.drop_duplicates()
print(len(all_rows), len(distinct_rows), len(all_rows) - len(distinct_rows))
```

</details>

In [ ]:
# Your code here


### 6.3 Subqueries

In SQL, a **subquery** is a query nested inside another query.

In Python, that often becomes:

- an **intermediate DataFrame** (list of keys), or
- a **scalar value** (one number), which you then use to filter, or
- a **per-group value** from a **correlated subquery**, often via `groupby().transform()` in pandas or `.over()` in Polars.

We will reuse the AP `invoices_pd` / `vendors_pd` tables already loaded.

#### Example A — `WHERE ... IN (SELECT ...)`

**SQL**
```sql
SELECT VendorName, VendorState
FROM Vendors
WHERE VendorID IN (
    SELECT VendorID
    FROM Invoices
    WHERE InvoiceTotal > 10000
);
```

The subquery finds vendor IDs with a large invoice; the outer query returns those vendors.

**pandas** — clear intermediate variable first

In [ ]:
large_invoice_vendor_ids = invoices_pd.loc[
    invoices_pd["InvoiceTotal"] > 10000, "VendorID"
]

vendors_with_large_invoices = vendors_pd[
    vendors_pd["VendorID"].isin(large_invoice_vendor_ids)
][["VendorName", "VendorState"]]

vendors_with_large_invoices.head()

Optional chained style (same idea, harder to debug):

In [ ]:
(
    vendors_pd[
        vendors_pd["VendorID"].isin(
            invoices_pd.loc[invoices_pd["InvoiceTotal"] > 10000, "VendorID"]
        )
    ][["VendorName", "VendorState"]]
.head())

**Polars**

In [ ]:
large_ids_pl = (
    invoices_pl.filter(pl.col("InvoiceTotal") > 10000)
    .select("VendorID")
)

(
    vendors_pl.join(large_ids_pl, on="VendorID", how="semi")
    .select("VendorName", "VendorState")
.head())

#### Example B — scalar subquery

**SQL**
```sql
SELECT InvoiceID, InvoiceTotal
FROM Invoices
WHERE InvoiceTotal > (
    SELECT AVG(InvoiceTotal) FROM Invoices
);
```

**pandas**

In [ ]:
avg_invoice_total = invoices_pd["InvoiceTotal"].mean()

above_average = invoices_pd[
    invoices_pd["InvoiceTotal"] > avg_invoice_total
][["InvoiceID", "InvoiceTotal"]]

print("Average InvoiceTotal:", round(avg_invoice_total, 2))
above_average.head()

**Polars**

In [ ]:
avg_total_pl = invoices_pl.select(pl.col("InvoiceTotal").mean()).item()

(
    invoices_pl.filter(pl.col("InvoiceTotal") > avg_total_pl)
    .select("InvoiceID", "InvoiceTotal")
.head())

#### Example C — correlated subquery

**SQL**
```sql
SELECT i.InvoiceID, i.VendorID, i.InvoiceTotal
FROM Invoices i
WHERE i.InvoiceTotal >
(
    SELECT AVG(i2.InvoiceTotal)
    FROM Invoices i2
    WHERE i2.VendorID = i.VendorID
);
```

A **correlated subquery** references a column from the outer query (`i.VendorID`). The inner `AVG` is computed **per vendor**, not for the whole table.

**pandas** — `transform()` computes one average per vendor for every row

In [ ]:
vendor_average = invoices_pd.groupby("VendorID")["InvoiceTotal"].transform("mean")

invoices_above_vendor_average = invoices_pd[
    invoices_pd["InvoiceTotal"] > vendor_average
][["InvoiceID", "VendorID", "InvoiceTotal"]]

invoices_above_vendor_average.head()

**Polars** — window expression with `.over("VendorID")`

In [ ]:
(
    invoices_pl
    .with_columns(
        pl.col("InvoiceTotal")
        .mean()
        .over("VendorID")
        .alias("VendorAverage")
    )
    .filter(pl.col("InvoiceTotal") > pl.col("VendorAverage"))
    .select("InvoiceID", "VendorID", "InvoiceTotal", "VendorAverage")
    .head()
)

**Check Your Understanding:** In one sentence, explain what a SQL subquery usually becomes when you rewrite it in pandas.

*Write a short comment or print statement in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
print("A SQL subquery usually becomes an intermediate Series/DataFrame of keys, "
      "or a scalar value, that you reuse in a later filter/join.")
```

</details>

In [ ]:
# Your code here


### 6.4 Recursive CTEs

Org charts are **hierarchical**: each employee may report to a manager, who reports to another manager.

SQL solves this with a **recursive CTE**.

**Important:** pandas and Polars do **not** provide SQL-style recursive CTE syntax. The Python below is an **equivalent algorithm**, not a direct syntax translation: we reproduce the recursive process using **repeated joins inside a Python loop**.

We will use this instructor-provided Employees data exactly:

| Column | Meaning |
|--------|---------|
| `EmployeeID` | Unique employee key |
| `LastName`, `FirstName` | Name parts |
| `DeptNo` | Department number (kept in the result for comparison) |
| `ManagerID` | EmployeeID of this person's manager (`NULL` = top of tree) |

In [ ]:
employee_data = [
    (1, "Smith", "Cindy", 2, None),
    (2, "Jones", "Elmer", 4, 1),
    (3, "Simonian", "Ralph", 2, 2),
    (4, "Hernandez", "Olivia", 1, 2),
    (5, "Aaronsen", "Robert", 2, 3),
    (6, "Watson", "Denise", 6, 3),
    (7, "Hardy", "Thomas", 5, 2),
    (8, "O'Leary", "Rhea", 4, 2),
    (9, "Locario", "Paulo", 6, 1),
]

employees_hier = pd.DataFrame(
    employee_data,
    columns=["EmployeeID", "LastName", "FirstName", "DeptNo", "ManagerID"],
)
employees_hier

**Original SQL recursive CTE (from the instructor)**

```sql
WITH EmployeesCTE AS
(
    -- Base case: employees with no manager (top of hierarchy)
    SELECT
        EmployeeID,
        LastName,
        FirstName,
        DeptNo,
        ManagerID,
        FirstName + ' ' + LastName AS EmployeeName,
        1 AS Rank
    FROM Employees
    WHERE ManagerID IS NULL

    UNION ALL

    -- Recursive case: employees who report to someone already in the CTE
    SELECT
        e.EmployeeID,
        e.LastName,
        e.FirstName,
        e.DeptNo,
        e.ManagerID,
        e.FirstName + ' ' + e.LastName AS EmployeeName,
        c.Rank + 1 AS Rank
    FROM Employees AS e
    JOIN EmployeesCTE AS c
      ON e.ManagerID = c.EmployeeID
)
SELECT *
FROM EmployeesCTE
ORDER BY Rank, EmployeeID;
```

- **Base case:** start with rows where `ManagerID IS NULL` and set `Rank = 1`
- **Recursive case:** find employees whose `ManagerID` matches an `EmployeeID` already found; set `Rank = manager Rank + 1`
- Repeat until no new reports are found
- Final result uses `FirstName + ' ' + LastName AS EmployeeName` and keeps `DeptNo` so you can compare to the original table

#### pandas: `build_employee_hierarchy()` — equivalent algorithm, not built-in syntax

`build_employee_hierarchy()` is a **custom function we write**, not a built-in pandas operation. It mirrors the recursive CTE step by step: start at the top, then keep attaching direct reports with joins in a `while` loop.

| Recursive CTE component | Python equivalent |
|-------------------------|-------------------|
| Anchor query | Select rows with `ManagerID.isna()` |
| Recursive query | Join employees to the previous level |
| `UNION ALL` | Append each discovered level |
| Stop condition | No additional employees are found |

In [ ]:
def build_employee_hierarchy(employees_df: pd.DataFrame) -> pd.DataFrame:
    """Build Ranked hierarchy like the recursive EmployeesCTE."""
    levels = []

    # Base case: no manager
    current = employees_df[employees_df["ManagerID"].isna()].copy()
    current["Rank"] = 1
    levels.append(current)

    # Recursive case: keep finding direct reports
    while True:
        managers = current[["EmployeeID", "Rank"]].rename(
            columns={"EmployeeID": "ManagerID", "Rank": "ManagerRank"}
        )
        nxt = employees_df.merge(managers, on="ManagerID", how="inner").copy()
        if nxt.empty:
            break
        nxt["Rank"] = nxt["ManagerRank"] + 1
        nxt = nxt.drop(columns=["ManagerRank"])
        levels.append(nxt)
        current = nxt

    hierarchy = pd.concat(levels, ignore_index=True)
    hierarchy["EmployeeName"] = (
        hierarchy["FirstName"] + " " + hierarchy["LastName"]
    )
    hierarchy = hierarchy.sort_values(["Rank", "EmployeeID"]).reset_index(drop=True)
    return hierarchy[
        ["Rank", "EmployeeID", "EmployeeName", "DeptNo", "ManagerID"]
    ]


hierarchy = build_employee_hierarchy(employees_hier)
display(hierarchy)
print("Ranks present:", sorted(hierarchy["Rank"].unique().tolist()))
assert sorted(hierarchy["Rank"].unique().tolist()) == [1, 2, 3, 4]

#### Polars note

Polars also has **no recursive CTE syntax**. Use the **same custom loop algorithm** (or convert the pandas result with `pl.DataFrame(hierarchy.to_dict(orient="list"))`). There is still no built-in recursive hierarchy helper — the loop is the translation of the CTE idea.

In [ ]:
# Optional: view the finished hierarchy as a Polars DataFrame
hierarchy_pl = pl.DataFrame(hierarchy.to_dict(orient="list"))
hierarchy_pl

**Practice:** Add a new employee who reports to **Robert Aaronsen** (find Robert's `EmployeeID` in the table). Rebuild the hierarchy and report the new employee's **Rank**.

Suggested new row: `EmployeeID=10`, `LastName="Nguyen"`, `FirstName="Alex"`, `DeptNo=2`, `ManagerID=<Robert's EmployeeID>`.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
robert_id = employees_hier.loc[
    (employees_hier["LastName"] == "Aaronsen")
    & (employees_hier["FirstName"] == "Robert"),
    "EmployeeID",
].iloc[0]

employees_with_new = pd.concat(
    [
        employees_hier,
        pd.DataFrame(
            [{
                "EmployeeID": 10,
                "LastName": "Nguyen",
                "FirstName": "Alex",
                "DeptNo": 2,
                "ManagerID": robert_id,
            }]
        ),
    ],
    ignore_index=True,
)

hierarchy_new = build_employee_hierarchy(employees_with_new)
new_rank = hierarchy_new.loc[hierarchy_new["EmployeeID"] == 10, "Rank"].iloc[0]
print("Robert EmployeeID:", robert_id)
print("Alex Nguyen Rank:", int(new_rank))
hierarchy_new.tail()
```

</details>

In [ ]:
# Your code here


### Advanced Section Recap

In **Section 6** you translated four advanced SQL topics into Python:

- **Outer joins** — keep unmatched rows from one or both tables (`LEFT JOIN`, `RIGHT JOIN`, `FULL OUTER JOIN`)
- **Set operations** — combine result sets row-by-row (`UNION`, `INTERSECT`, `EXCEPT`); SQL compares **complete rows**
- **Subqueries** — nested filters become intermediate DataFrames, scalars, or per-group values (`transform()` / `.over()`)
- **Recursive CTEs** — hierarchical walks become a **custom loop** with repeated joins (not built-in pandas/Polars syntax)

> **Quick lookup:** **Section 8 — Reference Guide** has the SQL ↔ **pandas** reference (with Polars shown for comparison). Use the **Pandas** column when completing required practice.

<hr style="border: 2px solid #003262">

## 7. Practice

Complete these problems in **Pandas** (required). Work through them in order — each builds on SQL skills you already have.

Data files: `Vendors.csv`, `Invoices.csv`, `Terms.csv`, `GLAccounts.csv`, `InvoiceLineItems.csv`

> **Optional:** After finishing a problem in Pandas, you may **Try this in Polars** using the matching `*_pl` DataFrames — but that is not required for the module.

**Problem 1.** Show only vendor names (`VendorName`) from the Vendors table.

<details>
<summary><strong>Solution:</strong></summary>

```python
vendors_pd[["VendorName"]].head()
# Polars: vendors_pl.select("VendorName").head()
```

</details>

In [ ]:
# Your solution here


**Problem 2.** Find invoices with `InvoiceTotal` greater than 10,000.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pd[invoices_pd["InvoiceTotal"] > 10000][["InvoiceID", "InvoiceTotal"]]
```

</details>

In [ ]:
# Your solution here


**Problem 3.** Sort invoices by `InvoiceTotal` from highest to lowest. Show the top 10.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pd.sort_values("InvoiceTotal", ascending=False).head(10)[["InvoiceID", "InvoiceTotal"]]
```

</details>

In [ ]:
# Your solution here


**Problem 4.** Find the average `InvoiceTotal` for each `TermsID`.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pd.groupby("TermsID")["InvoiceTotal"].mean().round(2)
```

</details>

In [ ]:
# Your solution here


**Problem 5.** Count how many vendors are in each `VendorState`. Sort by count descending.

<details>
<summary><strong>Solution:</strong></summary>

```python
vendors_pd.groupby("VendorState").size().sort_values(ascending=False)
```

</details>

In [ ]:
# Your solution here


**Problem 6.** Find the single invoice with the highest `InvoiceTotal`. Show `InvoiceID`, `VendorID`, and `InvoiceTotal`.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pd.loc[invoices_pd["InvoiceTotal"].idxmax(), ["InvoiceID", "VendorID", "InvoiceTotal"]]
```

</details>

In [ ]:
# Your solution here


**Problem 7.** Join `Invoices` and `Vendors` to show `VendorName`, `InvoiceNumber`, and `InvoiceTotal` for all invoices.

<details>
<summary><strong>Solution:</strong></summary>

```python
pd.merge(invoices_pd, vendors_pd[["VendorID", "VendorName"]], on="VendorID")[["VendorName", "InvoiceNumber", "InvoiceTotal"]].head()
```

</details>

In [ ]:
# Your solution here


**Problem 8.** How many invoices have a `PaymentDate` of `NULL` (unpaid)? Hint: use `.isna()`.

<details>
<summary><strong>Solution:</strong></summary>

```python
invoices_pd["PaymentDate"].isna().sum()
```

</details>

In [ ]:
# Your solution here


**Problem 9.** For each vendor state, what is the **total** invoice amount? (Join invoices to vendors first, then group by state.)

<details>
<summary><strong>Solution:</strong></summary>

```python
pd.merge(invoices_pd, vendors_pd[["VendorID", "VendorState"]], on="VendorID")
  .groupby("VendorState")["InvoiceTotal"].sum()
  .sort_values(ascending=False)
  .head(10)
```

</details>

In [ ]:
# Your solution here


**Problem 10.** Load `InvoiceLineItems.csv` and find the average line-item amount (`InvoiceLineItemAmount`) for each `AccountNo`. Join with `GLAccounts` to show `AccountDescription` too.

<details>
<summary><strong>Solution:</strong></summary>

```python
line_items = pd.read_csv("InvoiceLineItems.csv")
gl = pd.read_csv("GLAccounts.csv")
summary = line_items.groupby("AccountNo")["InvoiceLineItemAmount"].mean().reset_index()
pd.merge(summary, gl, on="AccountNo").sort_values("InvoiceLineItemAmount", ascending=False).head(10)
```

</details>

In [ ]:
# Your solution here


### Mini Challenge

The **AP manager** wants a report showing the **average invoice total for every payment term**, sorted from **highest average to lowest**, including the term description.

Expected columns: `TermsDescription`, `AvgInvoiceTotal`

Complete this in **Pandas** (required) in the cell below. Trying the same report in Polars is optional.

<details>
<summary><strong>Solution:</strong></summary>

```python
# Pandas solution
report = (
    pd.merge(invoices_pd, terms_pd, on="TermsID")
    .groupby(["TermsID", "TermsDescription"])["InvoiceTotal"]
    .mean()
    .reset_index(name="AvgInvoiceTotal")
    .sort_values("AvgInvoiceTotal", ascending=False)
)
report
```

</details>

In [ ]:
# Mini challenge — your solution


### Optional: Quick Visualization

Visualization is not the focus of this notebook, but a simple bar chart can make the challenge result easier to present.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(report["TermsDescription"], report["AvgInvoiceTotal"])
plt.xticks(rotation=15, ha="right")
plt.ylabel("Average Invoice Total ($)")
plt.title("Average Invoice Total by Payment Term")
plt.tight_layout()
plt.show()

<hr style="border: 2px solid #003262">

## 8. Reference Guide

Bookmark this page — return here when translating SQL to Python.

| Task | SQL | Pandas | Polars |
|------|-----|--------|--------|
| Load CSV | *(external)* | `pd.read_csv("file.csv")` | `pl.read_csv("file.csv")` |
| Preview rows | `SELECT TOP 5 *` | `df.head()` | `df.head()` |
| Select columns | `SELECT a, b` | `df[["a", "b"]]` | `df.select("a", "b")` |
| Filter | `WHERE x > 5` | `df[df["x"] > 5]` | `df.filter(pl.col("x") > 5)` |
| Sort | `ORDER BY x DESC` | `df.sort_values("x", ascending=False)` | `df.sort("x", descending=True)` |
| New column | `SELECT a + b AS c` | `df["c"] = df["a"] + df["b"]` | `df.with_columns((pl.col("a")+pl.col("b")).alias("c"))` |
| Mean | `AVG(x)` | `df["x"].mean()` | `df["x"].mean()` |
| Count | `COUNT(*)` | `df.groupby("g").size()` | `df.group_by("g").agg(pl.len())` |
| Group + agg | `GROUP BY g` | `df.groupby("g")["x"].mean()` | `df.group_by("g").agg(pl.col("x").mean())` |
| Distinct | `SELECT DISTINCT x` | `df["x"].unique()` | `df.select("x").unique()` |
| Join | `JOIN b ON a.id = b.id` | `pd.merge(a, b, on="id")` | `a.join(b, on="id")` |
| Outer join | `LEFT/RIGHT/FULL OUTER JOIN` | `pd.merge(..., how="left/right/outer")` | `join(..., how="left/full")`; reverse tables for RIGHT JOIN |
| Union all | `UNION ALL` | `pd.concat([a, b])` | `pl.concat([a, b])` |
| Union | `UNION` | `pd.concat(...).drop_duplicates()` | `pl.concat(...).unique()` |
| Intersect | `INTERSECT` | `merge` on distinct rows | `unique().join(..., how="inner")` |
| Except | `EXCEPT` | `merge` + `indicator` on distinct rows | `unique().join(..., how="anti")` |
| Correlated subquery | per-group filter | `.groupby().transform()` | `.over()` |
| Subquery IN | `WHERE x IN (SELECT ...)` | `.isin(keys)` | `join(..., how="semi")` |
| Null check | `WHERE x IS NULL` | `df["x"].isna()` | `pl.col("x").is_null()` |
| Row count | `COUNT(*)` | `len(df)` or `df.shape[0]` | `df.height` |

---

**Remember:** SQL retrieves data from databases. **Pandas** is the primary Python tool in this module; Polars is an optional modern comparison. Together, they form the bridge from databases to modern data science.

### Why Learn Polars?

> Reminder: Polars is **optional** in this module. Use this section for context, not as a second required syllabus.

**Performance.** Polars is written in Rust and uses parallel execution. On datasets with millions of rows, it is often significantly faster than Pandas.

**Modern data science.** New tools and courses (including updated Berkeley Data Science material) are introducing Polars alongside Pandas.

**Growing adoption.** Companies handling large-scale analytics increasingly evaluate Polars for ETL and exploratory analysis pipelines.

**Industry relevance.** SQL remains the language of databases; Pandas remains the most common Python tabular library; Polars is the fast, modern alternative worth knowing.

**Practical advice:** Master SQL first (done!). Focus on **Pandas** for this course. Revisit Polars later when you need speed or cleaner chained expressions — you do not need to learn SQL, Pandas, and Polars all at once.

<hr style="border: 2px solid #003262">

<hr style="border: 2px solid #C9B676">